# Laboratorio de Procesamiento de Videos en Español

Este notebook está diseñado para trabajar con videos de AWS *"This is My Architecture"* grabados en **español** u otros idiomas distintos al inglés.

El flujo del laboratorio realiza las siguientes tareas:
1. **Preparación de Datos:** Descarga de video en `data/raw` (si no existe), extracción local de audio en WAV y extracción de keyframes en `whiteboard_selection_lab/frames_new/VIDEO_ID`.
2. **Detección Automática de Idioma y Transcripción:** Carga del modelo Whisper `turbo`, análisis de los primeros 30 segundos del audio para auto-detectar el idioma y transcripción total guardada en local.
3. **Selección Inteligente de Pizarra:** Ejecución del selector inteligente (`frame_selector`) sobre los frames locales para encontrar la mejor toma libre de oclusión humana.
4. **Construcción y Comparación de Grafos (Cache):** Mapeo de los resultados de análisis de visión en cache (Standard y Parsimonious) a grafos de NetworkX y comparación directa contra el Ground Truth.
5. **Evaluación de Prompts e Integración de Idiomas (Gemini API):** Ejecución de un experimento en vivo con la API de Gemini Vision bajo prompts adaptados para traducir notas y descripciones del español al inglés.
6. **Comparación Detallada de Rendimiento:** Resumen interactivo en una tabla Pandas comparando el rendimiento de los tres grafos (Standard, Parsimonious y el de Prueba) contra el Ground Truth.

In [1]:
import cv2
import shutil
import os
import json
import sys
import torch
from pathlib import Path
import whisper

# ==========================================
# 1. CONFIGURACIÓN DEL VIDEO Y MODO
# ==========================================
VIDEO_ID = "-kA0ahrhX3I"
MODE = "parsimonious"  # "standard" o "parsimonious"

# Detectar el root del proyecto dinámicamente
PROJECT_ROOT = Path(os.path.abspath('')).parent if 'ipykernel' in sys.modules else Path(__file__).resolve().parent.parent
LAB_DIR = PROJECT_ROOT / 'whiteboard_selection_lab'

# Carpetas de trabajo locales en whiteboard_selection_lab
FRAMES_NEW_DIR = LAB_DIR / 'frames_new' / VIDEO_ID
TRANSCRIPTIONS_DIR = LAB_DIR / 'transcriptions'
LAB_WORKSPACE = LAB_DIR / 'lab_workspace' / VIDEO_ID
AUDIO_DIR = LAB_WORKSPACE / 'audio'

for d in (FRAMES_NEW_DIR, TRANSCRIPTIONS_DIR, LAB_WORKSPACE, AUDIO_DIR):
    d.mkdir(parents=True, exist_ok=True)

print(f"🚀 Iniciando preparación para el video ({MODE}): {VIDEO_ID}")


🚀 Iniciando preparación para el video (parsimonious): -3lnf5lzsH0


In [2]:
# ==========================================
# 2. DESCARGA Y EXTRACCIÓN DE AUDIO (Con Caché)
# ==========================================
transcript_file = TRANSCRIPTIONS_DIR / f"{VIDEO_ID}_transcript.json"
global_transcript = PROJECT_ROOT / 'data' / 'raw' / f"{VIDEO_ID}_transcript.json"
local_best_whiteboard = LAB_WORKSPACE / "best_whiteboard.jpg"
reports_best_whiteboard = PROJECT_ROOT / "cloudscape_reports" / VIDEO_ID / "best_whiteboard.jpg"
good_whiteboard_path = PROJECT_ROOT / "data" / "good_whiteboard" / f"{VIDEO_ID}.jpg"

has_cached_inputs = (transcript_file.exists() or global_transcript.exists()) and (local_best_whiteboard.exists() or reports_best_whiteboard.exists() or good_whiteboard_path.exists())

if has_cached_inputs:
    print("✅ Entradas de caché completas encontradas. Se omite descarga de video y audio.")
else:
    raw_video_path = PROJECT_ROOT / 'data' / 'raw' / f"{VIDEO_ID}.mp4"
    audio_path = AUDIO_DIR / f"{VIDEO_ID}.wav"

    # Descargar el video si no existe localmente
    if not raw_video_path.exists():
        print(f"📥 Video {VIDEO_ID}.mp4 no encontrado en raw. Descargándolo...")
        sys.path.append(str(PROJECT_ROOT))
        from scripts.core.downloader import download_video
        video_url = f"https://www.youtube.com/watch?v={VIDEO_ID}"
        download_video(video_url, output_dir=PROJECT_ROOT / 'data' / 'raw')
    else:
        print(f"✅ Video encontrado en: {raw_video_path}")

    # Extraer audio en PCM WAV 16kHz mono
    if not audio_path.exists():
        print(f"🔊 Extrayendo audio (FFmpeg) a {audio_path}...")
        os.system(f"ffmpeg -y -i {raw_video_path} -vn -acodec pcm_s16le -ar 16000 -ac 1 {audio_path}")
        print("✅ Audio extraído.")
    else:
        print(f"✅ El audio ya existe en: {audio_path}")


✅ Entradas de caché completas encontradas. Se omite descarga de video y audio.


In [3]:
# ==========================================
# 3. DETECCIÓN AUTOMÁTICA DE IDIOMA Y TRANSCRIPCIÓN (Whisper con Caché)
# ==========================================
transcript_file = TRANSCRIPTIONS_DIR / f"{VIDEO_ID}_transcript.json"
global_transcript = PROJECT_ROOT / 'data' / 'raw' / f"{VIDEO_ID}_transcript.json"

transcript_loaded = False

if transcript_file.exists():
    print(f"⚡ Transcripción encontrada en local del lab: {transcript_file}")
    transcript_loaded = True
elif global_transcript.exists():
    print(f"⚡ Transcripción encontrada en caché global. Copiando a local...")
    shutil.copy2(global_transcript, transcript_file)
    transcript_loaded = True

if not transcript_loaded:
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"🧠 Cargando modelo Whisper 'turbo' en {device}...")
    model = whisper.load_model("turbo", device=device)

    # Detección automática de idioma usando los primeros 30 segundos de audio
    print("🕵️ Detectando idioma del audio...")
    audio = whisper.load_audio(str(audio_path))
    audio = whisper.pad_or_trim(audio)
    # Usar model.dims.n_mels asegura que siempre coincida
    mel = whisper.log_mel_spectrogram(audio, n_mels=model.dims.n_mels).to(model.device)

    _, probs = model.detect_language(mel)
    detected_lang = max(probs, key=probs.get)
    print(f"✨ Idioma detectado automáticamente: '{detected_lang}' (Probabilidad: {probs[detected_lang]:.4f})")

    # Transcribir con Whisper usando el idioma detectado
    print("📝 Generando transcripción completa con Whisper...")
    result = model.transcribe(str(audio_path), language=detected_lang)

    # Formatear segmentos con marca de tiempo
    sys.path.append(str(PROJECT_ROOT))
    from scripts.core.transcriber import get_timestamped_segments
    segments = get_timestamped_segments(result)

    # Guardar en local del lab y en cache global raw
    with open(transcript_file, "w", encoding="utf-8") as f:
        json.dump(segments, f, indent=2, ensure_ascii=False)

    shutil.copy2(transcript_file, global_transcript)
    print(f"✅ Transcripción generada y guardada.")
else:
    print("✅ Se omitió la ejecución de Whisper por caché existente.")


⚡ Transcripción encontrada en local del lab: C:\Users\USUARIO\Downloads\AWS-Architecture-main\AWS-Architecture-main\whiteboard_selection_lab\transcriptions\-3lnf5lzsH0_transcript.json
✅ Se omitió la ejecución de Whisper por caché existente.


In [4]:
# ==========================================
# 4. EXTRACCIÓN DINÁMICA DE KEYFRAMES (Con Caché)
# ==========================================
local_best_whiteboard = LAB_WORKSPACE / "best_whiteboard.jpg"
reports_best_whiteboard = PROJECT_ROOT / "cloudscape_reports" / VIDEO_ID / "best_whiteboard.jpg"

skip_extraction = False
if local_best_whiteboard.exists():
    print(f"✅ Pizarra ganadora ya existe localmente en: {local_best_whiteboard}")
    skip_extraction = True
elif reports_best_whiteboard.exists():
    print(f"⚡ Pizarra ganadora encontrada en reportes. Copiando a local...")
    shutil.copy2(reports_best_whiteboard, local_best_whiteboard)
    skip_extraction = True

if not skip_extraction:
    cap = cv2.VideoCapture(str(raw_video_path))
    fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    umbral_80_porciento = int(total_frames * 0.80)

    intervalo_lento = int(fps * 10)  # Cada 10 segundos
    intervalo_rapido = int(fps * 1)   # Cada 1 segundo

    frame_count = 0
    saved_count = 0

    print(f"🎞️ Extrayendo keyframes... (FPS: {fps:.2f}, Total frames: {total_frames})")

    # Limpiar frames anteriores del lab para este video
    for f in FRAMES_NEW_DIR.glob('*.jpg'):
        f.unlink()

    while True:
        ret, frame = cap.read()
        if not ret:
            break
        
        intervalo_actual = intervalo_lento if frame_count < umbral_80_porciento else intervalo_rapido
        if frame_count % intervalo_actual == 0:
            cv2.imwrite(str(FRAMES_NEW_DIR / f"{VIDEO_ID}_frame_{frame_count}.jpg"), frame)
            saved_count += 1
        frame_count += 1
    cap.release()
    print(f"✅ Se guardaron {saved_count} frames en: {FRAMES_NEW_DIR}")
else:
    print("✅ Se omitió la extracción de keyframes por existir pizarra ganadora en caché/reportes.")


✅ Pizarra ganadora ya existe localmente en: C:\Users\USUARIO\Downloads\AWS-Architecture-main\AWS-Architecture-main\whiteboard_selection_lab\lab_workspace\-3lnf5lzsH0\best_whiteboard.jpg
✅ Se omitió la extracción de keyframes por existir pizarra ganadora en caché/reportes.


In [5]:
# ==========================================
# 5. SELECCIÓN DE PIZARRA INTELIGENTE
# ==========================================
sys.path.append(str(PROJECT_ROOT))
from scripts.core.frame_selector import select_best_frame

local_best_whiteboard = LAB_WORKSPACE / "best_whiteboard.jpg"

if local_best_whiteboard.exists():
    print(f"⚡ Pizarra ganadora cargada directamente desde caché/reportes: {local_best_whiteboard}")
else:
    print("🔍 Ejecutando el Frame Selector inteligente...")
    # Usamos la carpeta local frames_new del lab
    res_selector = select_best_frame(VIDEO_ID, frames_dir=LAB_DIR / 'frames_new', debug=True)

    # Copiar a lab_workspace para tener una referencia local cómoda de la pizarra ganadora
    best_frame_path = Path(res_selector['best_frame'])
    shutil.copy2(best_frame_path, local_best_whiteboard)

    print(f"🏆 Pizarra seleccionada: {best_frame_path.name}")
    print(f"📁 Copia local de pizarra ganadora guardada en: {local_best_whiteboard}")


⚡ Pizarra ganadora cargada directamente desde caché/reportes: C:\Users\USUARIO\Downloads\AWS-Architecture-main\AWS-Architecture-main\whiteboard_selection_lab\lab_workspace\-3lnf5lzsH0\best_whiteboard.jpg


## 7. DEFINICIÓN DE PROMPTS Y EJECUCIÓN CON GEMINI API (Con Pydantic)

Version en Ingles para trabajar de manera general


In [6]:
# ==========================================
# 7. DEFINICIÓN DE PROMPTS Y EJECUCIÓN CON GEMINI API (Con Pydantic)
# ==========================================
import base64
import json
import csv
import re
import networkx as nx
from google import genai
from pydantic import BaseModel
from config.settings import GEMINI_API_KEY, GEMINI_MODEL
from scripts.core.graph_builder import create_graph_from_cloudscape_json

if not GEMINI_API_KEY:
    raise ValueError("GEMINI_API_KEY no configurado. Revisa tu archivo .env")

client = genai.Client(api_key=GEMINI_API_KEY)

# 1. Cargar y codificar imagen de pizarra local en base64
ws_whiteboard = LAB_WORKSPACE / "best_whiteboard.jpg"
if not ws_whiteboard.exists():
    good_whiteboard_path = PROJECT_ROOT / "data" / "good_whiteboard" / f"{VIDEO_ID}.jpg"
    reports_best_whiteboard = PROJECT_ROOT / "cloudscape_reports" / VIDEO_ID / "best_whiteboard.jpg"
    if good_whiteboard_path.exists():
        shutil.copy2(good_whiteboard_path, ws_whiteboard)
    elif reports_best_whiteboard.exists():
        shutil.copy2(reports_best_whiteboard, ws_whiteboard)
    else:
        pizarra_dir = LAB_DIR / 'frames_new' / f"{VIDEO_ID}_pizarra"
        if (pizarra_dir / "best_whiteboard.jpg").exists():
            shutil.copy2(pizarra_dir / "best_whiteboard.jpg", ws_whiteboard)

image_bytes = ws_whiteboard.read_bytes()
image_b64 = base64.b64encode(image_bytes).decode("utf-8")

# 2. Cargar transcripción local
transcript_file = TRANSCRIPTIONS_DIR / f"{VIDEO_ID}_transcript.json"
with open(transcript_file, "r", encoding="utf-8") as f:
    segments = json.load(f)
transcript_text = " ".join(s.get("text", "").strip() for s in segments)

# 3. Cargar catálogo de servicios y actores
services_csv = PROJECT_ROOT / "graph_renderer" / "services.csv"
if not services_csv.exists():
    services_csv = PROJECT_ROOT / "data" / "cloudscape_gt" / "services.csv"

aws_services_list, user_actors_list = [], []
with open(services_csv, mode="r", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    for row in reader:
        name = row.get("name", "").strip()
        capability = row.get("capability", "").strip().lower()
        if not name:
            continue
        if capability == "user" or name.startswith("User"):
            user_actors_list.append(name)
        else:
            aws_services_list.append(name)

aws_services_str = ", ".join(sorted(aws_services_list))
user_actors_str = ", ".join(sorted(user_actors_list))

# ==========================================
# ESQUEMAS PYDANTIC
# ==========================================
class Entity(BaseModel):
    service: str
    name: str
    type: str
    rationale: str

class VisualConnection(BaseModel):
    source_label: str
    target_label: str
    arrow_direction: str
    description: str

class WorldModelSchema(BaseModel):
    entities: list[Entity]
    visual_connections: list[VisualConnection]

class GraphMetadata(BaseModel):
    name: str
    link: str
    categories: str
    graph_usable: bool
    notes: str

class Node(BaseModel):
    id: str
    service: str
    name: str
    notes: str

class Edge(BaseModel):
    source: str
    target: str
    flow_id: int
    seq: str
    type: str
    notes: str

class FinalArchitectureSchema(BaseModel):
    step_by_step_reasoning: str
    graph: GraphMetadata
    nodes: list[Node]
    edges: list[Edge]

ws_test_json = LAB_WORKSPACE / "test_analysis.json"
test_graph_path = LAB_WORKSPACE / "test_graph.graphml"

if MODE == "parsimonious":
    print("🔍 Modo seleccionado: Parsimonious (1 Fase con principio de parsimonia)")
    
    PARSIMONIOUS_PROMPT_TEMPLATE = """You are an expert AWS Solutions Architect. You are analyzing a whiteboard screenshot from an AWS "This is My Architecture" YouTube video, along with the full transcript of the video.

Your task is to extract the cloud architecture shown, encoding it using the Cloudscape dataset schema (FAST25 paper by Satija et al.) in a way that matches the style and level of detail of the manual ground truth dataset as closely as possible.

## RULES:
1. Use SHORT AWS service names for `service` field: e.g., "S3", "Lambda", "EC2", "DynamoDB", "EKS", "CloudFront", etc.

2. USER ACTOR NORMALIZATION: Only add User nodes that are explicitly shown or mentioned. Choose EXCLUSIVELY from this list: <USER_ACTORS_PLACEHOLDER>. 
   - Map external consumers to web/mobile users.
   - For internal staff, default to "UserCompanyDeveloper" for ANY technical, engineering, security, or operations teams. Reserve "UserCompanyAnalyst" strictly for non-technical business analysts, and "UserCompanyAgent" for customer support/call center agents.

3. COMPUTE NORMALIZATION: Map rendering engine clusters, ASGs, or application instances running on EC2 directly to the "EC2" service, putting context like "Rendering Engines" or "ASG" in the name or notes field.

4. THIRD-PARTY & OPEN SOURCE AVOIDANCE OF HALLUCINATIONS: 
   - Do NOT guess AWS managed services. If a generic open-source technology (e.g., MySQL, Kafka, Cassandra) or custom software is drawn or mentioned without explicitly naming the AWS managed equivalent (like RDS or MSK), you MUST map it to "ThirdParty" (or "EC2" if explicitly self-hosted). 
   - Map corporate networks or physical legacy infrastructure to "OnPremDC".

5. AUDIO-VISUAL BALANCE (RETAIN HIDDEN CORE SERVICES): The whiteboard physical drawing is the primary guide, but you MUST explicitly include distinct backend services described in the transcript as driving the architecture's logic. Pay special attention to observability tools, security auditing, messaging/queueing decouplers, or automation triggers that might only be represented by a small badge, an arrow, or implied by the flow.

6. Edges must have: flow_id (integer), seq (string), type ("data" or "meta"). Default to "data" for all edges.

7. STRICT UNIDIRECTIONAL EDGES (NO RETURN PATHS): Map strictly the primary, forward-moving flow of requests, data, or triggers. **STRICTLY PROHIBITED:** Do not add return paths, response edges, bidirectional arrows, or API acknowledgments. If Service A initiates contact with Service B, draw exactly one edge: A -> B.

8. CORE CONNECTIVITY: Compress complex, multi-step back-and-forth communications between two components into a single directed edge representing the main logical intent.

9. The `notes` field for nodes should capture context from the transcript: how the service is used.

10. AVOID TRANSIENT ARTIFACTS: Do NOT create nodes for transient artifacts, machine images, config templates, or zip files. Represent these as descriptions on the edges connecting permanent components.

## PARSIMONY PRINCIPLE (STRICT DEDUPLICATION ONLY):
Your goal is a clean graph, but you MUST NEVER omit distinct functional AWS services. 
- "Parsimony" STRICTLY means deduplicating multiple instances of the SAME service (e.g., if there are 3 EC2 instances doing the same logical job, combine them into 1 EC2 node). 
- It NEVER means deleting functionally distinct services. Whether a service acts as a core compute processor, a peripheral data source, a messaging broker, or an automation trigger, if it is drawn or explicitly stated to actively route, store, or process data, YOU MUST INCLUDE IT.

## VALID SERVICE NAMES:
You MUST only use names from this list of canonical services when defining the `service` field in the nodes list:
<AWS_SERVICES_PLACEHOLDER>

## OUTPUT FORMAT:
Return ONLY valid JSON (no markdown fences):
{
  "step_by_step_reasoning": "Analyze the transcript chronologically...",
  "graph": {
    "name": "<title of the architecture>",
    "link": "<youtube URL if known, else empty string>",
    "categories": "<comma-separated from: data_ingestion, interactive, compute_intensive, control, other>",
    "graph_usable": true,
    "notes": "<distilled context>"
  },
  "nodes": [
    {"id": "0", "service": "...", "name": "", "notes": "..."}
  ],
  "edges": [
    {"source": "0", "target": "1", "flow_id": 0, "seq": "0", "type": "data", "notes": ""}
  ]
}"""
    
    formatted_prompt = PARSIMONIOUS_PROMPT_TEMPLATE.replace(
        "<USER_ACTORS_PLACEHOLDER>", user_actors_str
    ).replace(
        "<AWS_SERVICES_PLACEHOLDER>", aws_services_str
    )
    full_prompt = f"{formatted_prompt}\n\n## FULL TRANSCRIPT:\n{transcript_text}"
    
    if ws_test_json.exists():
        print("⚡ Cargando análisis Parsimonious desde caché local...")
        with open(ws_test_json, "r", encoding="utf-8") as f:
            test_result = json.load(f)
    else:
        print(f"Sending single-phase query to Gemini Vision (Parsimonious prompt)... ({GEMINI_MODEL})")
        response = client.models.generate_content(
            model=GEMINI_MODEL,
            contents=[
                {
                    "parts": [
                        {"text": full_prompt},
                        {
                            "inline_data": {
                                "mime_type": "image/jpeg",
                                "data": image_b64,
                            }
                        },
                    ]
                }
            ],
            config={
                "response_mime_type": "application/json",
                "response_schema": FinalArchitectureSchema,
            },
        )
        test_result = json.loads(response.text)
        ws_test_json.write_text(json.dumps(test_result, indent=2, ensure_ascii=False), encoding="utf-8")
        print(f"✓ Análisis Parsimonious final guardado en: {ws_test_json}")
        
    test_graph = create_graph_from_cloudscape_json(test_result, video_id=VIDEO_ID)
    nx.write_graphml(test_graph, str(test_graph_path))
    print(f"✓ Grafo experimental Parsimonious exportado localmente a: {test_graph_path}")

else:
    print("🔍 Modo seleccionado: Standard (2 Fases Modeler-Planner)")
    ws_world_model_json = LAB_WORKSPACE / "world_model.json"
    
    from scripts.core.vision_analyzer import MFR_STAGE_1_PROMPT_TEMPLATE, MFR_STAGE_2_PROMPT_TEMPLATE
    
    # --- FASE 1: Agente Modelador ---
    if ws_world_model_json.exists():
        print(f"⚡ Cargando World Model desde caché local...")
        with open(ws_world_model_json, "r", encoding="utf-8") as f:
            world_model = json.load(f)
    else:
        print("🔍 Generando nuevo Modelo del Mundo con Gemini Vision...")
        formatted_prompt_1 = MFR_STAGE_1_PROMPT_TEMPLATE.replace(
            "<USER_ACTORS_PLACEHOLDER>", user_actors_str
        ).replace(
            "<AWS_SERVICES_PLACEHOLDER>", aws_services_str
        )
        full_prompt_1 = f"{formatted_prompt_1}\n\n## FULL TRANSCRIPT:\n{transcript_text}"
        response_1 = client.models.generate_content(
            model=GEMINI_MODEL,
            contents=[
                {
                    "parts": [
                        {"text": full_prompt_1},
                        {
                            "inline_data": {
                                "mime_type": "image/jpeg",
                                "data": image_b64,
                            }
                        },
                    ]
                }
            ],
            config={
                "response_mime_type": "application/json",
                "response_schema": WorldModelSchema,
            },
        )
        world_model = json.loads(response_1.text)
        ws_world_model_json.write_text(json.dumps(world_model, indent=2, ensure_ascii=False), encoding="utf-8")
        print(f"✓ Modelo del Mundo guardado en: {ws_world_model_json}")
        
    # --- FASE 2: Agente Planificador ---
    if ws_test_json.exists():
        print(f"⚡ Cargando Análisis Final local desde caché...")
        with open(ws_test_json, "r", encoding="utf-8") as f:
            test_result = json.load(f)
    else:
        print("🔍 Generando nuevo Análisis Final con Gemini y Pydantic...")
        formatted_prompt_2 = MFR_STAGE_2_PROMPT_TEMPLATE.replace(
            "<WORLD_MODEL_PLACEHOLDER>", json.dumps(world_model, indent=2, ensure_ascii=False)
        ).replace(
            "<AWS_SERVICES_PLACEHOLDER>", aws_services_str
        ).replace(
            "<USER_ACTORS_PLACEHOLDER>", user_actors_str
        )
        full_prompt_2 = f"{formatted_prompt_2}\n\n## FULL TRANSCRIPT:\n{transcript_text}"
        response_2 = client.models.generate_content(
            model=GEMINI_MODEL,
            contents=[
                {
                    "parts": [
                        {"text": full_prompt_2},
                        {
                            "inline_data": {
                                "mime_type": "image/jpeg",
                                "data": image_b64,
                            }
                        },
                    ]
                }
            ],
            config={
                "response_mime_type": "application/json",
                "response_schema": FinalArchitectureSchema,
            },
        )
        test_result = json.loads(response_2.text)
        ws_test_json.write_text(json.dumps(test_result, indent=2, ensure_ascii=False), encoding="utf-8")
        print(f"✓ Análisis de visión final guardado en: {ws_test_json}")
        
    # Construir y exportar el grafo experimental
    test_graph = create_graph_from_cloudscape_json(test_result, video_id=VIDEO_ID)
    nx.write_graphml(test_graph, str(test_graph_path))
    print(f"✓ Grafo experimental Standard exportado localmente a: {test_graph_path}")


🔍 Modo seleccionado: Parsimonious (1 Fase con principio de parsimonia)
Sending single-phase query to Gemini Vision (Parsimonious prompt)... (gemini-2.5-flash)
✓ Análisis Parsimonious final guardado en: C:\Users\USUARIO\Downloads\AWS-Architecture-main\AWS-Architecture-main\whiteboard_selection_lab\lab_workspace\-kA0ahrhX3I\test_analysis.json
✓ Graph created: 8 nodes, 8 edges
✓ Grafo experimental Parsimonious exportado localmente a: C:\Users\USUARIO\Downloads\AWS-Architecture-main\AWS-Architecture-main\whiteboard_selection_lab\lab_workspace\-kA0ahrhX3I\test_graph.graphml


## Resumen Comparativo de Rendimiento Lado a Lado

La celda final realiza una comparación detallada de rendimiento calculando la precisión, recall y F1 de los servicios y las conexiones de la nueva prueba frente a las ejecuciones estándar y parsimoniosas cached, mostrándolo en una tabla de Pandas interactiva.

In [7]:
# ==========================================
# 8. COMPARACIÓN DETALLADA LADO A LADO
# ==========================================
import pandas as pd
from scripts.utils.evaluate_graphs import evaluate_pair, load_services_catalog

# Configurar pandas para que no trunque columnas en la impresión de texto
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

# Cargar catálogo de servicios
services_csv = PROJECT_ROOT / "graph_renderer" / "services.csv"
if not services_csv.exists():
    services_csv = PROJECT_ROOT / "data" / "cloudscape_gt" / "services.csv"
catalog_dict = load_services_catalog(services_csv)

# Cargar grafos
gt_path = PROJECT_ROOT / "data" / "cloudscape_gt" / f"{VIDEO_ID}.graphml"
RAW_DIR = PROJECT_ROOT / "data" / "raw"
gt_g = nx.read_graphml(str(gt_path))
test_g = nx.read_graphml(str(test_graph_path))

# Rutas locales de grafos en data/
ws_std_graph = RAW_DIR / f"{VIDEO_ID}.graphml"
if not ws_std_graph.exists():
    ws_std_graph = PROJECT_ROOT / "data" / "graphs" / f"{VIDEO_ID}.graphml"

ws_parsi_graph = RAW_DIR / f"{VIDEO_ID}_parsimonious.graphml"
if not ws_parsi_graph.exists():
    ws_parsi_graph = PROJECT_ROOT / "data" / "graphs_parsimonious" / f"{VIDEO_ID}.graphml"

# Evaluar nueva prueba contra GT
test_eval = evaluate_pair(test_g, gt_g, VIDEO_ID, catalog_dict)

def evaluar_previo(path):
    if path.exists():
        try:
            g = nx.read_graphml(str(path))
            return evaluate_pair(g, gt_g, VIDEO_ID, catalog_dict)
        except Exception as e:
            return {"error": str(e)}
    return None

std_eval = evaluar_previo(ws_std_graph)
parsi_eval = evaluar_previo(ws_parsi_graph)

# Generar filas de DataFrame comparativo
filas = []
def add_row(label, key, is_pct=True):
    fila = {"Métrica": label}
    if key == "gen_nodes":
        fila["Ground Truth"] = gt_g.number_of_nodes()
    elif key == "gen_edges":
        fila["Ground Truth"] = gt_g.number_of_edges()
    else:
        fila["Ground Truth"] = "—"
        
    # Standard
    if std_eval and key in std_eval:
        val = std_eval[key]
        fila["Standard Original"] = f"{val:.1%}" if is_pct else val
    else:
        fila["Standard Original"] = "N/A"
        
    # Parsimonious
    if parsi_eval and key in parsi_eval:
        val = parsi_eval[key]
        fila["Parsimonious Original"] = f"{val:.1%}" if is_pct else val
    else:
        fila["Parsimonious Original"] = "N/A"
        
    # Test
    if key in test_eval:
        val = test_eval[key]
        fila["Nueva Prueba (Test)"] = f"{val:.1%}" if is_pct else val
    else:
        fila["Nueva Prueba (Test)"] = "—"
        
    filas.append(fila)

add_row("Número de Nodos", "gen_nodes", is_pct=False)
add_row("Número de Aristas", "gen_edges", is_pct=False)
add_row("Service F1 (Unique)", "svc_f1")
add_row("Service Precision", "svc_precision")
add_row("Service Recall", "svc_recall")
add_row("Edge F1 (Connections)", "edge_f1")
add_row("Edge Precision", "edge_precision")
add_row("Edge Recall", "edge_recall")

df_resumen = pd.DataFrame(filas)
print("📊 RESUMEN COMPARATIVO DE RENDIMIENTO DE GRAFOS DE PRUEBA:")
display(df_resumen)

# Detalles cualitativos de errores
print("\n🔍 ERRORES O DETALLES EN LA NUEVA PRUEBA (TEST):")
print(f"  • Servicios Faltantes (Omitidos):   {test_eval.get('services_missing', [])}")
print(f"  • Servicios Alucinados (Inventados): {test_eval.get('services_hallucinated', [])}")

if parsi_eval:
    print("\n🔄 COMPARACIÓN DE ERRORES CON PARSIMONIOUS ORIGINAL:")
    print(f"  • Faltaban antes: {parsi_eval.get('services_missing', [])}")
    print(f"  • Faltan ahora:   {test_eval.get('services_missing', [])}")
    print(f"  • Alucinaba antes: {parsi_eval.get('services_hallucinated', [])}")
    print(f"  • Alucina ahora:   {test_eval.get('services_hallucinated', [])}")


📊 RESUMEN COMPARATIVO DE RENDIMIENTO DE GRAFOS DE PRUEBA:
                Métrica Ground Truth  Standard Original  Parsimonious Original  Nueva Prueba (Test)
        Número de Nodos            9                 11                     11                    8
      Número de Aristas            8                  4                     10                    8
    Service F1 (Unique)            —              85.7%                  85.7%               100.0%
      Service Precision            —              85.7%                  85.7%               100.0%
         Service Recall            —              85.7%                  85.7%               100.0%
  Edge F1 (Connections)            —              66.7%                  55.6%                25.0%
         Edge Precision            —             100.0%                  50.0%                25.0%
            Edge Recall            —              50.0%                  62.5%                25.0%

🔍 ERRORES O DETALLES EN LA NUEVA PRUEBA (

In [8]:
import networkx as nx
from pathlib import Path
from rich.console import Console
from rich.table import Table
import os, sys

# 1. Configuración del video a comparar
VIDEO_ID = "-kA0ahrhX3I"
PROJECT_ROOT = Path(os.path.abspath('')).parent if 'ipykernel' in sys.modules else Path(__file__).resolve().parent.parent

# 2. Rutas de los grafos
std_path = PROJECT_ROOT / "data" / "graphs" / f"{VIDEO_ID}.graphml"
parsi_path = PROJECT_ROOT / "data" / "graphs_parsimonious" / f"{VIDEO_ID}.graphml"

# 3. Validar existencia y cargar
if not std_path.exists():
    print(f"❌ No se encontró el grafo standard en: {std_path}")
elif not parsi_path.exists():
    print(f"❌ No se encontró el grafo parsimonious en: {parsi_path}")
else:
    std_g = nx.read_graphml(str(std_path))
    parsi_g = nx.read_graphml(str(parsi_path))

    console = Console()

    # Helpers para extraer información limpia
    def get_node_info(G):
        return {n: (attrs.get("service", "N/A"), attrs.get("name", "N/A")) for n, attrs in G.nodes(data=True)}

    std_nodes = get_node_info(std_g)
    parsi_nodes = get_node_info(parsi_g)

    def get_edges_info(G, nodes_dict):
        edges = []
        for src, tgt, attrs in G.edges(data=True):
            src_svc = nodes_dict.get(src, ("?", "?"))[0]
            tgt_svc = nodes_dict.get(tgt, ("?", "?"))[0]
            flow_id = attrs.get("flow_id", "0")
            seq = attrs.get("seq", "0")
            etype = attrs.get("type", "data")
            edges.append((src_svc, tgt_svc, flow_id, seq, etype))
        return edges

    std_edges = get_edges_info(std_g, std_nodes)
    parsi_edges = get_edges_info(parsi_g, parsi_nodes)

    # ─── TABLA 1: RESUMEN GENERAL ───
    resumen_table = Table(title=f"Resumen Comparativo - Video: {VIDEO_ID}", border_style="cyan", show_lines=True)
    resumen_table.add_column("Métrica", style="bold")
    resumen_table.add_column("Standard (Original)", style="green")
    resumen_table.add_column("Parsimonious (Simplificado)", style="yellow")

    resumen_table.add_row("Número de Nodos", str(std_g.number_of_nodes()), str(parsi_g.number_of_nodes()))
    resumen_table.add_row("Número de Aristas", str(std_g.number_of_edges()), str(parsi_g.number_of_edges()))

    std_flows = len(set(attrs.get("flow_id", 0) for _, _, attrs in std_g.edges(data=True)))
    parsi_flows = len(set(attrs.get("flow_id", 0) for _, _, attrs in parsi_g.edges(data=True)))
    resumen_table.add_row("Número de Workflows (flow_id)", str(std_flows), str(parsi_flows))

    console.print(resumen_table)

    # ─── TABLA 2: COMPARACIÓN DE SERVICIOS (NODOS) ───
    console.print("\n[bold cyan]🔍 Comparación de Nodos (Servicios):[/]")
    std_svc_set = set(svc for svc, _ in std_nodes.values())
    parsi_svc_set = set(svc for svc, _ in parsi_nodes.values())

    only_in_std_svc = std_svc_set - parsi_svc_set
    only_in_parsi_svc = parsi_svc_set - std_svc_set
    both_svc = std_svc_set & parsi_svc_set

    nodes_table = Table(border_style="cyan", show_lines=True)
    nodes_table.add_column("Servicios en Ambos", style="bold white")
    nodes_table.add_column("Solo en Standard", style="green")
    nodes_table.add_column("Solo en Parsimonious", style="yellow")

    max_len = max(len(both_svc), len(only_in_std_svc), len(only_in_parsi_svc))
    both_list = sorted(list(both_svc))
    std_only_list = sorted(list(only_in_std_svc))
    parsi_only_list = sorted(list(only_in_parsi_svc))

    for i in range(max_len):
        b = both_list[i] if i < len(both_list) else ""
        s = std_only_list[i] if i < len(std_only_list) else ""
        p = parsi_only_list[i] if i < len(parsi_only_list) else ""
        nodes_table.add_row(b, s, p)

    console.print(nodes_table)

    # ─── TABLA 3: COMPARACIÓN DE ARISTAS (CONEXIONES) ───
    console.print("\n[bold cyan]🔍 Comparación de Conexiones (Flujo de Datos):[/]")

    def edge_to_str(edge):
        src, tgt, fid, seq, etype = edge
        return f"{src} ──[{etype}]──> {tgt}  (Flow: {fid}, Seq: {seq})"

    std_edges_str = set(edge_to_str(e) for e in std_edges)
    parsi_edges_str = set(edge_to_str(e) for e in parsi_edges)

    only_in_std_edges = std_edges_str - parsi_edges_str
    only_in_parsi_edges = parsi_edges_str - std_edges_str
    both_edges = std_edges_str & parsi_edges_str

    edges_table = Table(border_style="cyan", show_lines=True)
    edges_table.add_column("Conexiones en Ambos", style="bold white")
    edges_table.add_column("Solo en Standard", style="green")
    edges_table.add_column("Solo en Parsimonious", style="yellow")

    max_edges_len = max(len(both_edges), len(only_in_std_edges), len(only_in_parsi_edges))
    both_edges_list = sorted(list(both_edges))
    std_edges_only_list = sorted(list(only_in_std_edges))
    parsi_edges_only_list = sorted(list(only_in_parsi_edges))

    for i in range(max_edges_len):
        be = both_edges_list[i] if i < len(both_edges_list) else ""
        se = std_edges_only_list[i] if i < len(std_edges_only_list) else ""
        pe = parsi_edges_only_list[i] if i < len(parsi_edges_only_list) else ""
        edges_table.add_row(be, se, pe)

    console.print(edges_table)


                   Resumen Comparativo - Video: -3lnf5lzsH0                    
┌───────────────────────────┬─────────────────────┬───────────────────────────┐
│                           │                     │ Parsimonious              │
│ Métrica                   │ Standard (Original) │ (Simplificado)            │
├───────────────────────────┼─────────────────────┼───────────────────────────┤
│ Número de Nodos           │ 14                  │ 9                         │
├───────────────────────────┼─────────────────────┼───────────────────────────┤
│ Número de Aristas         │ 15                  │ 9                         │
├───────────────────────────┼─────────────────────┼───────────────────────────┤
│ Número de Workflows       │ 3                   │ 2                         │
│ (flow_id)                 │                     │                           │
└───────────────────────────┴─────────────────────┴───────────────────────────┘

🔍 Comparación de Nodos (Servicios):
┌──